In [2]:
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL = "https://store.steampowered.com/charts/topsellers/KR/2025-12-23"

def get_top100_appids(url: str, target=100):
    opts = Options()
    # opts.add_argument("--headless=new")  # 창 안 띄우려면 주석 해제
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--disable-gpu")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

    driver = webdriver.Chrome(options=opts)
    try:
        driver.get(url)

        wait = WebDriverWait(driver, 10)

        # 테이블(혹은 차트 루트)이 뜰 때까지 대기
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))

        # 스크롤하면서 appid 누적 수집
        seen = []
        seen_set = set()

        def harvest():
            links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/app/']")
            for a in links:
                href = a.get_attribute("href") or ""
                m = re.search(r"/app/(\d+)/", href)
                if m:
                    appid = m.group(1)
                    if appid not in seen_set:
                        seen_set.add(appid)
                        seen.append(appid)

        # 초기 수집
        time.sleep(1)
        harvest()

        # 스크롤 루프
        last_count = -1
        same_count_rounds = 0

        while len(seen) < target and same_count_rounds < 6:
            # 페이지 아래로 스크롤 (가상리스트 렌더 유도)
            driver.execute_script("window.scrollBy(0, 900);")
            time.sleep(0.6)
            harvest()

            # 변화 감지
            if len(seen) == last_count:
                same_count_rounds += 1
            else:
                same_count_rounds = 0
                last_count = len(seen)

        return seen[:target]

    finally:
        driver.quit()

if __name__ == "__main__":
    appids = get_top100_appids(URL, target=100)
    print("count:", len(appids))
    print(appids)


ModuleNotFoundError: No module named 'selenium'

In [ ]:
import time
import random
import requests
import pandas as pd

SECONDS_IN_DAY = 86400

# ✅ 세션 + 헤더 (안정성 ↑)
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://store.steampowered.com/",
})

def safe_get(url, params=None, timeout=30, max_retries=8):
    """
    502/5xx, 429 등으로 끊겨도 자동 재시도하는 GET
    """
    base_backoff = 1.5

    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(url, params=params, timeout=timeout)

            # ✅ 429: 레이트리밋
            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")
                wait = int(retry_after) if retry_after and retry_after.isdigit() else 10
                wait += random.uniform(0, 1.0)
                print(f"[429] rate limited -> wait {wait:.1f}s (attempt {attempt}/{max_retries})")
                time.sleep(wait)
                continue

            # ✅ 5xx: 서버/게이트웨이(502 포함) 불안정
            if 500 <= r.status_code < 600:
                wait = min(60, base_backoff * (2 ** (attempt - 1))) + random.uniform(0, 0.7)
                print(f"[{r.status_code}] server error -> wait {wait:.1f}s (attempt {attempt}/{max_retries})")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r

        except requests.RequestException as e:
            wait = min(60, base_backoff * (2 ** (attempt - 1))) + random.uniform(0, 0.7)
            print(f"[EXC] {type(e).__name__}: {e} -> wait {wait:.1f}s (attempt {attempt}/{max_retries})")
            time.sleep(wait)

    raise RuntimeError(f"safe_get failed after {max_retries} retries: {url}")

def fetch_reviews_last_n_days(
    appid: int,
    days: int = 180,                 # 기간 (일)
    filter: str = "recent",
    language: str = "all",
    review_type: str = "all",
    purchase_type: str = "all",
    num_per_page: int = 100,
    filter_offtopic_activity: int = 1,
    sleep_sec: float = 0.5           # ✅ 너무 빠르면 502/429 더 잘 뜸
):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    cursor = "*"
    rows = []
    page = 0

    now_ts = int(time.time())
    cutoff_ts = now_ts - days * SECONDS_IN_DAY

    while True:
        params = {
            "json": 1,
            "filter": filter,
            "language": language,
            "review_type": review_type,
            "purchase_type": purchase_type,
            "num_per_page": num_per_page,
            "cursor": cursor,
            "filter_offtopic_activity": filter_offtopic_activity,
        }

        # ✅ 여기만 교체: requests.get -> safe_get
        r = safe_get(url, params=params, timeout=30, max_retries=8)
        data = r.json()

        if data.get("success") != 1:
            raise RuntimeError(f"API success != 1: {data}")

        reviews = data.get("reviews", [])
        if not reviews:
            break

        stop = False
        for rev in reviews:
            ts_created = rev.get("timestamp_created")

            # 기간 자르기
            if ts_created is not None and ts_created < cutoff_ts:
                stop = True
                break

            author = rev.get("author", {})
            rows.append({
                "appid": appid,  # ✅ 여기서 바로 appid 넣어버리기 (제일 깔끔)
                "recommendationid": rev.get("recommendationid"),
                "steamid": author.get("steamid"),
                "num_games_owned": author.get("num_games_owned"),
                "num_reviews_author": author.get("num_reviews"),
                "playtime_forever": author.get("playtime_forever"),
                "playtime_last_two_weeks": author.get("playtime_last_two_weeks"),
                "playtime_at_review": author.get("playtime_at_review"),
                "deck_playtime_at_review": author.get("deck_playtime_at_review"),
                "last_played": author.get("last_played"),
                "language": rev.get("language"),
                "review": rev.get("review"),
                "timestamp_created": ts_created,
                "timestamp_updated": rev.get("timestamp_updated"),
                "voted_up": rev.get("voted_up"),
                "votes_up": rev.get("votes_up"),
                "votes_funny": rev.get("votes_funny"),
                "weighted_vote_score": rev.get("weighted_vote_score"),
                "comment_count": rev.get("comment_count"),
                "steam_purchase": rev.get("steam_purchase"),
                "received_for_free": rev.get("received_for_free"),
                "written_during_early_access": rev.get("written_during_early_access"),
                "developer_response": rev.get("developer_response"),
                "timestamp_dev_responded": rev.get("timestamp_dev_responded"),
                "primarily_steam_deck": rev.get("primarily_steam_deck"),
            })

        page += 1
        print(f"appid={appid}, page={page}, fetched={len(reviews)}, kept={len(rows)}")

        if stop:
            break

        cursor = data.get("cursor")
        time.sleep(sleep_sec)

    return pd.DataFrame(rows)

app_id_list = [
    '730', '1049590', '578080', '3564740', '3557620', '2139460', '1808500', '960170',
    '2344520', '2879840', '1245620', '1086940', '1771300', '1091500', '2246340',
    '3513350', '3240220', '985810', '1903340', '2426960', '1623730', '236390',
    '3527290', '216150', '2827200', '381210', '2807960', '413150', '261550',
    '2622380', '1426210', '1973530', '3609080', '294100', '1174180', '1172470',
    '2444750', '648800', '3241660', '2592160', '2001120', '3489700', '3551340',
    '3405690', '3932890', '728880', '553850', '1984270', '3159330', '4077430',
    '108600', '440', '990080', '1145350', '2138330', '230410', '1621690',
    '3167020', '3472040', '394360', '1326470', '1449850', '1158310', '1144200',
    '1222140', '2215430', '227300', '3059520', '1030300', '2993780', '3101040',
    '1222670', '2927200', '934700', '570', '286160', '835570', '814380',
    '1778820', '1260320', '1142710', '1825750', '526870', '1435790', '1562700',
    '3447040', '1868140', '1627720', '2651280', '1203620', '1966720', '2183900',
    '2948190', '1929290', '2436940', '2968420', '2322010', '703080', '1551360',
    '1888160'
]

if __name__ == "__main__":
    dfs = []
    failed = []

    for appid in app_id_list:
        try:
            df = fetch_reviews_last_n_days(appid, days=180)
            print(f"done {appid}: {df.shape}")
            dfs.append(df)

            # ✅ 게임 하나 끝나면 잠깐 쉬어주기(안정성↑)
            time.sleep(1.0)

        except Exception as e:
            print(f"❌ fail {appid}: {e}")
            failed.append(appid)
            time.sleep(5)

    if dfs:
        final_df = pd.concat(dfs, ignore_index=True)
        final_df.to_csv("steam_reviews_last180d.csv", index=False, encoding="utf-8-sig")
        print("total rows:", final_df.shape)
    else:
        print("No data collected.")

    print("FAILED APPIDS:", failed)


appid=730, page=1, fetched=100, kept=100
appid=730, page=2, fetched=100, kept=200
appid=730, page=3, fetched=100, kept=300
appid=730, page=4, fetched=100, kept=400
appid=730, page=5, fetched=100, kept=500
appid=730, page=6, fetched=100, kept=600
appid=730, page=7, fetched=100, kept=700
appid=730, page=8, fetched=100, kept=800
appid=730, page=9, fetched=100, kept=900
appid=730, page=10, fetched=100, kept=1000
appid=730, page=11, fetched=100, kept=1100
appid=730, page=12, fetched=100, kept=1200
appid=730, page=13, fetched=100, kept=1300
appid=730, page=14, fetched=100, kept=1400
appid=730, page=15, fetched=100, kept=1500
appid=730, page=16, fetched=100, kept=1600
appid=730, page=17, fetched=100, kept=1700
appid=730, page=18, fetched=100, kept=1800
appid=730, page=19, fetched=100, kept=1900
appid=730, page=20, fetched=100, kept=2000
appid=730, page=21, fetched=100, kept=2100
appid=730, page=22, fetched=100, kept=2200
appid=730, page=23, fetched=100, kept=2300
appid=730, page=24, fetched=1